# ✈️ Airline Passenger Satisfaction Prediction

## 03. Data Preprocessing

This notebook prepares the Airline Passenger Satisfaction dataset for Artificial Neural Network modeling.

The preprocessing workflow includes:

- Loading the training and testing datasets
- Removing non-predictive identifier columns
- Handling missing values
- Encoding categorical features
- Encoding the target variable
- Splitting the training data into training and validation sets
- Scaling numerical features
- Saving the processed datasets and preprocessing objects

In [1]:
# Step 2 : Importing Libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Step 3 : Loading the Datasets

train_path = "../data/raw/train.csv"
test_path = "../data/raw/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)

Training Dataset Shape: (103904, 25)
Testing Dataset Shape: (25976, 25)


In [3]:
# Step 4 : Separating Features and Target

target_column = "satisfaction"

X = train_df.drop(columns=[target_column])
y = train_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

print("Training Features Shape:", X.shape)
print("Training Target Shape:", y.shape)

print("Testing Features Shape:", X_test.shape)
print("Testing Target Shape:", y_test.shape)

Training Features Shape: (103904, 24)
Training Target Shape: (103904,)
Testing Features Shape: (25976, 24)
Testing Target Shape: (25976,)


In [4]:
# Step 5 : Removing Identifier Columns

identifier_columns = [
    "Unnamed: 0",
    "id"
]

X = X.drop(columns=identifier_columns)
X_test = X_test.drop(columns=identifier_columns)

print("Training Features Shape:", X.shape)
print("Testing Features Shape:", X_test.shape)

Training Features Shape: (103904, 22)
Testing Features Shape: (25976, 22)


In [5]:
# Step 6 : Encoding the Target Variable

target_mapping = {
    "neutral or dissatisfied": 0,
    "satisfied": 1
}

y = y.map(target_mapping)
y_test = y_test.map(target_mapping)

print("Encoded Training Target:")
print(y.value_counts())

print("\nEncoded Testing Target:")
print(y_test.value_counts())

Encoded Training Target:
satisfaction
0    58879
1    45025
Name: count, dtype: int64

Encoded Testing Target:
satisfaction
0    14573
1    11403
Name: count, dtype: int64


In [6]:
# Step 7 : Identifying Numerical and Categorical Features

categorical_features = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

numerical_features = [
    column
    for column in X.select_dtypes(include=np.number).columns
]

print("Categorical Features:")
print(categorical_features)

print("\nNumber of Categorical Features:", len(categorical_features))

print("\nNumerical Features:")
print(numerical_features)

print("\nNumber of Numerical Features:", len(numerical_features))

Categorical Features:
['Gender', 'Customer Type', 'Type of Travel', 'Class']

Number of Categorical Features: 4

Numerical Features:
['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

Number of Numerical Features: 18


In [7]:
# Step 8 : Creating Preprocessing Pipelines

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [8]:
# Step 9 : Combining Preprocessing Pipelines

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [9]:
# Step 10 : Creating Training and Validation Sets

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Validation Set:", X_val.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation Target Distribution:")
print(y_val.value_counts(normalize=True).round(3))

Training Set: (83123, 22)
Validation Set: (20781, 22)

Training Target Distribution:
satisfaction
0    0.567
1    0.433
Name: proportion, dtype: float64

Validation Target Distribution:
satisfaction
0    0.567
1    0.433
Name: proportion, dtype: float64


In [10]:
# Step 11 : Fitting the Preprocessor

preprocessor.fit(X_train)

print("Preprocessor fitted successfully.")

Preprocessor fitted successfully.


In [11]:
# Step 12 : Transforming the Datasets

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Validation Shape:", X_val_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (83123, 27)
Processed Validation Shape: (20781, 27)
Processed Testing Shape: (25976, 27)


In [12]:
# Step 13 : Converting Processed Data to NumPy Arrays

X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_val_processed = np.asarray(X_val_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

y_train = np.asarray(y_train, dtype=np.float32)
y_val = np.asarray(y_val, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

print("Training Features:", X_train_processed.shape)
print("Validation Features:", X_val_processed.shape)
print("Testing Features:", X_test_processed.shape)

Training Features: (83123, 27)
Validation Features: (20781, 27)
Testing Features: (25976, 27)


In [13]:
# Step 14 : Checking Remaining Missing Values

print(
    "Missing Values in Training:",
    np.isnan(X_train_processed).sum()
)

print(
    "Missing Values in Validation:",
    np.isnan(X_val_processed).sum()
)

print(
    "Missing Values in Testing:",
    np.isnan(X_test_processed).sum()
)

Missing Values in Training: 0
Missing Values in Validation: 0
Missing Values in Testing: 0


In [14]:
# Step 15 : Checking Feature Scaling

print(
    "Minimum Training Feature Value:",
    X_train_processed.min()
)

print(
    "Maximum Training Feature Value:",
    X_train_processed.max()
)

print(
    "Mean of Training Features:",
    X_train_processed.mean()
)

print(
    "Standard Deviation of Training Features:",
    X_train_processed.std()
)

Minimum Training Feature Value: -3.098204
Maximum Training Feature Value: 40.992275
Mean of Training Features: 0.14814815
Standard Deviation of Training Features: 0.89043075


In [15]:
# Step 16 : Final Target Distribution

print("Training Target Distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation Target Distribution:")
print(pd.Series(y_val).value_counts())

print("\nTesting Target Distribution:")
print(pd.Series(y_test).value_counts())

Training Target Distribution:
0.0    47103
1.0    36020
Name: count, dtype: int64

Validation Target Distribution:
0.0    11776
1.0     9005
Name: count, dtype: int64

Testing Target Distribution:
0.0    14573
1.0    11403
Name: count, dtype: int64


In [16]:
# Step 17 : Getting Final Feature Names

feature_names = preprocessor.get_feature_names_out()

print("Total Processed Features:", len(feature_names))

print("\nFirst 20 Processed Feature Names:")
print(feature_names[:20])

Total Processed Features: 27

First 20 Processed Feature Names:
['numerical__Age' 'numerical__Flight Distance'
 'numerical__Inflight wifi service'
 'numerical__Departure/Arrival time convenient'
 'numerical__Ease of Online booking' 'numerical__Gate location'
 'numerical__Food and drink' 'numerical__Online boarding'
 'numerical__Seat comfort' 'numerical__Inflight entertainment'
 'numerical__On-board service' 'numerical__Leg room service'
 'numerical__Baggage handling' 'numerical__Checkin service'
 'numerical__Inflight service' 'numerical__Cleanliness'
 'numerical__Departure Delay in Minutes'
 'numerical__Arrival Delay in Minutes' 'categorical__Gender_Female'
 'categorical__Gender_Male']


In [ ]:
# Step 18 : Saving Processed Training Data

np.save(
    "../data/processed/X_train.npy",
    X_train_processed
)

np.save(
    "../data/processed/y_train.npy",
    y_train
)

print("Training data saved successfully.")

Training data saved successfully.


In [ ]:
# Step 19 : Saving Processed Validation Data

np.save(
    "../data/processed/X_val.npy",
    X_val_processed
)

np.save(
    "../data/processed/y_val.npy",
    y_val
)

print("Validation data saved successfully.")

Validation data saved successfully.


In [ ]:
# Step 20 : Saving Processed Test Data

np.save(
    "../data/processed/X_test.npy",
    X_test_processed
)

np.save(
    "../data/processed/y_test.npy",
    y_test
)

print("Testing data saved successfully.")

Testing data saved successfully.


In [ ]:
# Step 21 : Saving the Preprocessing Pipeline

joblib.dump(
    preprocessor,
    "../data/processed/preprocessor.joblib"
)

print("Preprocessing pipeline saved successfully.")

Preprocessing pipeline saved successfully.


In [ ]:
# Step 22 : Saving Processed Feature Names

np.save(
    "../data/processed/feature_names.npy",
    feature_names
)

print("Feature names saved successfully.")

Feature names saved successfully.


In [ ]:
# Step 23 : Verifying Saved Files

import os

processed_files = [
    "X_train.npy",
    "y_train.npy",
    "X_val.npy",
    "y_val.npy",
    "X_test.npy",
    "y_test.npy",
    "preprocessor.joblib",
    "feature_names.npy"
]

for file_name in processed_files:
    file_path = os.path.join(
        "../data/processed",
        file_name
    )

    print(
        f"{file_name}:",
        "Exists" if os.path.exists(file_path) else "Missing"
    )

X_train.npy: Exists
y_train.npy: Exists
X_val.npy: Exists
y_val.npy: Exists
X_test.npy: Exists
y_test.npy: Exists
preprocessor.joblib: Exists
feature_names.npy: Exists


In [ ]:
# Step 24 : Final Preprocessing Summary

print("=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)

print("\nOriginal Training Rows:", len(train_df))
print("Original Testing Rows:", len(test_df))

print("\nTraining Samples:", X_train_processed.shape[0])
print("Validation Samples:", X_val_processed.shape[0])
print("Testing Samples:", X_test_processed.shape[0])

print("\nFinal ANN Input Features:", X_train_processed.shape[1])

print("\nTraining Target Shape:", y_train.shape)
print("Validation Target Shape:", y_val.shape)
print("Testing Target Shape:", y_test.shape)

print("\nMissing Values After Preprocessing:")
print("Training:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_val_processed).sum())
print("Testing:", np.isnan(X_test_processed).sum())

print("\nPreprocessing completed successfully.")

FINAL PREPROCESSING SUMMARY

Original Training Rows: 103904
Original Testing Rows: 25976

Training Samples: 83123
Validation Samples: 20781
Testing Samples: 25976

Final ANN Input Features: 27

Training Target Shape: (83123,)
Validation Target Shape: (20781,)
Testing Target Shape: (25976,)

Missing Values After Preprocessing:
Training: 0
Validation: 0
Testing: 0

Preprocessing completed successfully.


# ✅ Preprocessing Conclusion

The Airline Passenger Satisfaction dataset has been successfully prepared for Artificial Neural Network modeling.

### Preprocessing performed:

- Removed non-predictive identifier columns.
- Separated features and target variable.
- Encoded the target into binary values.
- Handled missing numerical values using median imputation.
- Handled categorical missing values using most-frequent imputation.
- Applied One-Hot Encoding to categorical features.
- Applied StandardScaler to numerical features.
- Created stratified training and validation datasets.
- Kept the provided test dataset separate for final evaluation.
- Prevented data leakage by fitting the preprocessing pipeline only on the training data.
- Saved the processed datasets and preprocessing pipeline for future model training and deployment.

The resulting processed data is now ready for Artificial Neural Network development.